# Qualitative comparison figure (`fig:qualitative`, `figures/qualitative_comparison.pdf`)

Frames of rendered tracking videos: DexYCB (`20200918-subject-06__20200918_114117`, view_00, t = 12, 24) and
Panoptic Studio (`tennis`, camera 14, t = 90, 150), one column per method.

Needs the renders (not in this repository). Make them with the Tracktail code (https://github.com/AI-HHMI/tracktail, branch `paper`):
1. predictions on the 100-track selections: `cotracker_baseline/cotracker_100.sh`, `mvtracker_baseline/mvtracker_100.sh`,
   `arrow_baseline/arrow_100.sh`, `scripts/tracktail_100.sh` (canonical cameras, `pose3d_cached_100.npz`);
2. `scripts/render_qualitative_gtvis.sh`: renders every method with the ground-truth visibility, into
   `<RENDERS_ROOT>/{cotracker,mvtracker,arrow,tracktail,gt}/{dex_ycb,cmupanoptic_3dgs}/<trial>/<view>.mp4`.

Then `RENDERS_ROOT=<dir> jupyter nbconvert --to notebook --execute figure_qualitative.ipynb`.
(Moved here from `format_arrow.ipynb` cell 7; `archive/figure_qualitative_comparison.ipynb` is the older version.)

In [ ]:
import os

import cv2
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec

import paths   # input/output locations, set by environment variables (see paths.py)

matplotlib.rcParams.update({'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix'})
FIG_OUT = str(paths.FIG_OUT)
os.makedirs(FIG_OUT, exist_ok=True)

In [ ]:
dex_trial = '20200918-subject-06__20200918_114117'
cmu_trial = 'tennis'
# renders of scripts/render_qualitative_gtvis.sh (Tracktail code): every method shows/hides a point by the
# ground-truth visibility, so all columns show the same points at every frame
G = str(paths.require(paths.RENDERS_ROOT, 'RENDERS_ROOT'))   # <RENDERS_ROOT>/<method>/<dataset>/<trial>/<view>.mp4
video_prefixes = {
    ds: {'CoTracker3 Baseline': f'{G}/cotracker/{d}/{trial}',
         'MVTracker':           f'{G}/mvtracker/{d}/{trial}',
         'ARROW':               f'{G}/arrow/{d}/{trial}',
         'Tracktail':           f'{G}/tracktail/{d}/{trial}',
         'Ground Truth':        f'{G}/gt/{d}/{trial}'}   # last, next to Tracktail
    for ds, d, trial in [('dex-ycb', 'dex_ycb', dex_trial), ('cmupanoptic_3dgs', 'cmupanoptic_3dgs', cmu_trial)]
}
methods = list(video_prefixes['dex-ycb'])
specs = [{'dataset': 'dex-ycb', 'views': ['view_00'], 'frames': [11, 23]},
         {'dataset': 'cmupanoptic_3dgs', 'views': ['14'], 'frames': [89, 149]}]


def read_frame(path, frame_num, target_ar=16 / 9):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ok, img = cap.read()
    cap.release()
    if not ok:
        raise FileNotFoundError(f'cannot read frame {frame_num} of {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]   # center-crop to 16:9 so all rows share the same aspect ratio
    if w / h < target_ar:
        new_h = int(round(w / target_ar)); y0 = (h - new_h) // 2; img = img[y0:y0 + new_h]
    elif w / h > target_ar:
        new_w = int(round(h * target_ar)); x0 = (w - new_w) // 2; img = img[:, x0:x0 + new_w]
    return img


fs = 20
sns.set_style('ticks', {'font.family': 'STIXGeneral'})
n_cols = len(methods)
fig = plt.figure(figsize=(3.5 * n_cols, 10))
gs = gridspec.GridSpec(4, n_cols, wspace=0.0, hspace=0.0, left=0.05, right=0.95)
axes = [[fig.add_subplot(gs[row, col]) for col in range(n_cols)] for row in range(4)]
for row_idx, row_axes in enumerate(axes):
    spec = specs[row_idx // 2]
    frame_num = spec['frames'][row_idx % 2]
    for col_idx, ax in enumerate(row_axes):
        method = methods[col_idx]
        ax.imshow(read_frame(os.path.join(video_prefixes[spec['dataset']][method], f"{spec['views'][0]}.mp4"), frame_num))
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if row_idx == 0:
            ax.set_title(method, fontsize=fs)
        if col_idx == 0:
            ax.set_ylabel(f't = {frame_num + 1}', fontsize=fs - 1)

for dataset_idx, dataset_name in enumerate(['Dex YCB', 'Panoptic Studio']):
    top, bottom = axes[2 * dataset_idx][0].get_position(), axes[2 * dataset_idx + 1][0].get_position()
    fig.text(0.015, (top.y0 + bottom.y1) / 2, dataset_name, ha='center', va='center', rotation=90,
             fontsize=fs + 2, transform=fig.transFigure)
fig.savefig(os.path.join(FIG_OUT, 'qualitative_comparison.pdf'), dpi=800, bbox_inches='tight')